In [1]:
import numpy as np
from scipy.linalg import lstsq
import matplotlib.pyplot as plt

def gmres_basic(A, b, x0=None, max_iter=100, tol=1e-8):
    """
    Basic implementation of the GMRES algorithm solving Ax = b.
    Solves the least-squares problem min || beta * e1 - H_bar * y ||_2 
    at each step using a standard least squares solver.
    
    Parameters:
    A        : ndarray, the coefficient matrix (n x n)
    b        : ndarray, the right-hand side vector (n,)
    x0       : ndarray, initial guess (default is zero vector)
    max_iter : int, maximum number of iterations
    tol      : float, tolerance for the residual norm
    
    Returns:
    x        : ndarray, the estimated solution
    res_hist : list, history of the residual norms
    """
   # Get the size of the matrix A
    n = A.shape[0]
    
    # If no initial guess is provided, start with a vector of zeros
    if x0 is None:
        x0 = np.zeros(n)
        
    # # Initial residual
    A_times_x0 = np.dot(A, x0)
    r0 = b - A_times_x0
    
    # Calculate the norm (length) of the initial residual
    beta = np.linalg.norm(r0)
    
    # Keep track of the residual norms for plotting later
    res_hist = [beta]
    
    # If the initial guess is already good enough
    if beta < tol:
        return x0, res_hist
    
    # Set up empty matrices to hold calculations
    # V holds the basis vectors
    V = np.zeros((n, max_iter + 1))
    # H holds the Hessenberg matrix values
    H = np.zeros((max_iter + 1, max_iter))
    
    # The first column of V is the normalized initial residual
    V[:, 0] = r0 / beta
    
    # To count numbers of iterations
    iterations_run = 0
    
    for m in range(max_iter):
        iterations_run = m + 1
        
        # Arnoldi Iteration (Modified Gram-Schmidt)
        # Multiply A by the most recent basis vector in V
        v_next = np.dot(A, V[:, m])
        
        # Orthogonalize v_next against all previous vectors
        for j in range(m + 1):
            H[j, m] = np.dot(V[:, j], v_next)
            v_next = v_next - (H[j, m] * V[:, j])
            
        # The subdiagonal element is the norm of the new vector
        H[m + 1, m] = np.linalg.norm(v_next)
        
        # Prevent division by zero if the exact solution has been found
        if H[m + 1, m] != 0:
            V[:, m + 1] = v_next / H[m + 1, m]
            
        # Set up the least squares problem
        # Create a vector that is zero everywhere except the first element
        e1 = np.zeros(m + 2)
        e1[0] = beta
        
        # Slice the H matrix to get the currently filled portion
        H_bar = H[0 : m + 2, 0 : m + 1]
        
        # 3. Solve the least squares problem
        # lstsq returns a tuple of 4 things. We only want the first thing (index 0), 
        # which is the solution vector 'y'.
        least_squares_result = lstsq(H_bar, e1)
        y = least_squares_result[0]
        
        # Calculate how close to the true answer
        H_bar_times_y = np.dot(H_bar, y)
        residual_vector = H_bar_times_y - e1
        current_res = np.linalg.norm(residual_vector)
        
        res_hist.append(current_res)
        
        # Stop looping if it has reached the tolerance
        if current_res < tol:
            break
            
    # Compute the final answer
    # We only use the columns of V that we actually filled
    V_filled = V[:, 0 : iterations_run]
    V_times_y = np.dot(V_filled, y)
    
    # Final solution
    x = x0 + V_times_y
    
    return x, res_hist



# Example Test Case 
if __name__ == "__main__":
    # Create a 50x50 test matrix
    n = 50
    np.random.seed(42)
    A = np.random.rand(n, n) + (n * np.eye(n))
    b = np.random.rand(n)
    
    # Run function
    x, res_history = gmres_basic(A, b, max_iter=n)
    
    # Plot convergence
    plt.figure(figsize=(8, 5))
    plt.semilogy(res_history, marker='o', linestyle='-', color='b')
    plt.title('Basic GMRES Convergence')
    plt.xlabel('Iteration')
    plt.ylabel('Residual Norm')
    plt.grid(True, which="both", ls="--")
    plt.show()
    
    # Print results
    total_iterations = len(res_history) - 1
    final_residual = res_history[-1]
    
    print("Converged in", total_iterations, "iterations.")
    print("Final residual norm:", final_residual)
    

NameError: name 'norm' is not defined